# Module 6 – Advanced RAG & GraphRAG
## GraphRAG Knowledge Explorer with Neo4j

**Assignment goal:** Build a GraphRAG system that:
1. Extracts entities and relationships from a knowledge-rich document using an Ollama LLM.
2. Stores the extracted knowledge as nodes and relationships in Neo4j.
3. Performs vector similarity retrieval.
4. Performs 1–2 hop graph traversal.
5. Combines graph and vector retrieval using Reciprocal Rank Fusion (RRF).
6. Answers at least five multi-hop questions.
7. Compares Vector RAG with GraphRAG.

**Stack:** Python/Jupyter, Ollama + Qwen, Neo4j, LangChain, FAISS, Ollama embeddings, Pydantic.

> Before running: start Ollama and Neo4j. Install/pull the models configured below.

## 1. Install dependencies

Run this once in the notebook environment.

In [ ]:
%pip install -qU neo4j langchain langchain-community langchain-core langchain-ollama langchain-text-splitters faiss-cpu pydantic pandas tabulate

## 2. Imports and configuration

Defaults:
- Chat model: `qwen3:1.7b`
- Embedding model: `nomic-embed-text`
- Neo4j: `bolt://localhost:7687`

Change these values or set environment variables if required.

In [ ]:
import os
import re
from pathlib import Path
from typing import List

import pandas as pd
from pydantic import BaseModel, Field

from neo4j import GraphDatabase
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHAT_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "qwen3:1.7b")
EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "nomic-embed-text")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
DOCUMENT_PATH = DATA_DIR / "document.txt"

print("Chat model:", CHAT_MODEL)
print("Embedding model:", EMBED_MODEL)
print("Neo4j URI:", NEO4J_URI)
print("Document:", DOCUMENT_PATH)

## 3. Create a knowledge-rich sample document

The notebook creates `data/document.txt` automatically. You can later replace it with your own Wikipedia/company/scientific document.

In [ ]:
sample_document = """
TECHNOLOGY COMPANIES AND THEIR CONNECTIONS

Tesla and Elon Musk

Tesla, Inc. is an electric vehicle and clean energy company. Elon Musk joined Tesla as an early investor and became its chief executive officer. Tesla is headquartered in Austin, Texas. Tesla develops electric vehicles, battery storage products, and solar energy products. The company is strongly associated with the electric vehicle industry and renewable energy.

Elon Musk is also associated with SpaceX. SpaceX is an aerospace company founded by Elon Musk. SpaceX is headquartered in Hawthorne, California. SpaceX develops launch vehicles and spacecraft and operates the Starlink satellite internet system. Starlink is a satellite communications concept connected to SpaceX.

PayPal and X

Before his work at Tesla and SpaceX, Elon Musk was involved with X.com, an online financial company. X.com later became part of PayPal. PayPal is a financial technology company headquartered in San Jose, California. Peter Thiel was a co-founder of PayPal and served as its chief executive officer during an early period. The PayPal network connected several entrepreneurs who later became influential in technology.

Zip2

Elon Musk and his brother Kimbal Musk co-founded Zip2. Zip2 developed online city guides and business directories. Compaq acquired Zip2 in 1999. The Zip2 experience helped Elon Musk build experience in technology entrepreneurship and online services.

SolarCity

SolarCity was a solar energy services company founded by Lyndon Rive and Peter Rive, with support from Elon Musk. SolarCity was headquartered in San Mateo, California. The company installed and financed solar energy systems. Tesla acquired SolarCity in 2016. SolarCity therefore became connected to Tesla through the acquisition and to renewable energy through its solar business.

Twitter and X

Elon Musk acquired Twitter in 2022. Twitter was a social media platform. After the acquisition, Twitter was rebranded as X. X is therefore connected to Elon Musk through the acquisition and to Twitter through the rebranding. The company is associated with social media, online communication, and digital services.

Locations and relationships

Austin is a major technology center in Texas and is the headquarters location of Tesla. Hawthorne is a city in California and is the headquarters location of SpaceX. San Jose is a city in California and is the headquarters location of PayPal. San Mateo is a city in California and is the headquarters location of SolarCity.

These companies illustrate how people, organizations, locations, and concepts can form an interconnected knowledge graph. A question may require following several relationships. For example, a question can start with a person, move to a company, then move to a headquarters city, and finally move to another organization or founder connected with that location.

Additional technology context

Electric vehicles use electric powertrains and batteries rather than conventional internal combustion engines as their primary propulsion system. Battery storage supports renewable energy by storing electricity for later use. Solar energy converts sunlight into electricity. Satellite internet uses satellites to provide network connectivity over large geographic areas. These concepts are connected to the technology companies described above, but relationships should only be extracted when the document explicitly supports them.
""".strip()

DOCUMENT_PATH.write_text(sample_document, encoding="utf-8")
print("Created:", DOCUMENT_PATH)
print("Characters:", len(sample_document))
print(sample_document[:1200])

## 4. Load and chunk the document

In [ ]:
text = DOCUMENT_PATH.read_text(encoding="utf-8")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = splitter.split_text(text)

print("Document characters:", len(text))
print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks, 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk[:500])

## 5. Initialize Ollama

Run these outside Jupyter if necessary:

```bash
ollama pull qwen3:1.7b
ollama pull nomic-embed-text
ollama list
```

In [ ]:
llm = ChatOllama(
    model=CHAT_MODEL,
    temperature=0
)

embeddings = OllamaEmbeddings(
    model=EMBED_MODEL
)

print("Ollama clients created.")

## 6. Test Ollama and embeddings

In [ ]:
test_response = llm.invoke("Reply with exactly: Ollama connection successful")
print(test_response.content)

test_embedding = embeddings.embed_query("GraphRAG test sentence")
print("Embedding dimension:", len(test_embedding))

## 7. Connect to Neo4j

For Neo4j Desktop, `bolt://localhost:7687` is commonly used. For Neo4j Aura, replace the URI and credentials.

In [ ]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()
print("Neo4j connection successful.")

## 8. Define structured extraction schemas

In [ ]:
class Entity(BaseModel):
    name: str = Field(description="Canonical entity name")
    type: str = Field(description="One of Person, Organization, Location, Concept")

class Relationship(BaseModel):
    source: str = Field(description="Source entity name")
    relationship: str = Field(description="Short relationship such as CEO_OF")
    target: str = Field(description="Target entity name")

class GraphExtraction(BaseModel):
    entities: List[Entity]
    relationships: List[Relationship]

class QueryEntities(BaseModel):
    entities: List[str]

## 9. LLM entity and relationship extraction

In [ ]:
extraction_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a precise knowledge graph extraction system.

Extract entities and explicit relationships from the text.

Allowed entity types:
- Person
- Organization
- Location
- Concept

Rules:
1. Extract only facts explicitly supported by the text.
2. Do not invent facts.
3. Use canonical names when possible.
4. Relationship names must be short uppercase identifiers using underscores.
5. Only include relationships supported by the supplied text.
6. Return all useful entities and relationships from the chunk.
"""
    ),
    ("human", "{text}")
])

structured_llm = llm.with_structured_output(GraphExtraction)

def extract_graph(text_chunk: str) -> GraphExtraction:
    chain = extraction_prompt | structured_llm
    return chain.invoke({"text": text_chunk})

test_extraction = extract_graph(chunks[0])
print(test_extraction.model_dump_json(indent=2))

## 10. Extract from every chunk

In [ ]:
extractions = []

for i, chunk in enumerate(chunks, 1):
    try:
        result = extract_graph(chunk)
        extractions.append(result)
        print(
            f"Chunk {i}/{len(chunks)}: "
            f"{len(result.entities)} entities, "
            f"{len(result.relationships)} relationships"
        )
    except Exception as exc:
        print(f"Chunk {i} failed: {exc}")

print("Successful chunks:", len(extractions))

## 11. Normalize entities and triples

In [ ]:
def clean_relationship(value: str) -> str:
    value = value.strip().upper()
    value = re.sub(r"[^A-Z0-9_]", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "RELATED_TO"

def normalize_entity_type(value: str) -> str:
    allowed = {"PERSON", "ORGANIZATION", "LOCATION", "CONCEPT"}
    value = value.strip().upper()
    return value if value in allowed else "CONCEPT"

entities_by_name = {}
triples = set()

for result in extractions:
    for entity in result.entities:
        name = entity.name.strip()
        if name:
            entities_by_name[name.lower()] = {
                "name": name,
                "type": normalize_entity_type(entity.type)
            }

    for rel in result.relationships:
        source = rel.source.strip()
        target = rel.target.strip()
        relationship = clean_relationship(rel.relationship)
        if source and target:
            triples.add((source, relationship, target))

print("Unique entities:", len(entities_by_name))
print("Unique relationships:", len(triples))

print("\nSample entities:")
for item in list(entities_by_name.values())[:15]:
    print(item)

print("\nSample triples:")
for triple in list(triples)[:20]:
    print(triple)

## 12. Reset the Neo4j graph

**Warning:** This deletes all nodes and relationships in the connected database. Use a dedicated assignment database.

In [ ]:
RESET_GRAPH = True

if RESET_GRAPH:
    driver.execute_query("MATCH (n) DETACH DELETE n")
    print("Neo4j graph reset.")
else:
    print("Neo4j graph was not reset.")

## 13. Create uniqueness constraint

In [ ]:
driver.execute_query("""
CREATE CONSTRAINT entity_name_unique IF NOT EXISTS
FOR (e:Entity)
REQUIRE e.name IS UNIQUE
""")
print("Entity uniqueness constraint ready.")

## 14. Load entity nodes

In [ ]:
node_rows = list(entities_by_name.values())

driver.execute_query(
    """
    UNWIND $rows AS row
    MERGE (e:Entity {name: row.name})
    SET e.type = row.type
    """,
    rows=node_rows
)

print(f"Loaded {len(node_rows)} entity nodes.")

## 15. Load relationship edges

In [ ]:
for source, relationship, target in triples:
    query = f"""
    MATCH (a:Entity {{name: $source}})
    MATCH (b:Entity {{name: $target}})
    MERGE (a)-[:{relationship}]->(b)
    """
    driver.execute_query(
        query,
        source=source,
        target=target
    )

print(f"Loaded {len(triples)} relationships.")

## 16. Verify the graph

In [ ]:
records = driver.execute_query(
    """
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.name AS source,
           a.type AS source_type,
           type(r) AS relationship,
           b.name AS target,
           b.type AS target_type
    ORDER BY source
    LIMIT 100
    """
).records

graph_df = pd.DataFrame([dict(r) for r in records])
graph_df

## 17. Neo4j Browser visualization

Open Neo4j Browser and run:

```cypher
MATCH (n)-[r]->(m)
RETURN n, r, m
```

For Elon Musk's 1–2 hop neighborhood:

```cypher
MATCH p=(n:Entity {name: 'Elon Musk'})-[*1..2]-(m)
RETURN p
```

These queries are useful screenshots for the assignment.

## 18. Build the vector index

In [ ]:
vector_store = FAISS.from_texts(
    chunks,
    embedding=embeddings
)

print("FAISS vector index created for", len(chunks), "chunks.")

## 19. Vector retrieval

In [ ]:
def vector_retrieval(query: str, k: int = 5) -> list[str]:
    docs = vector_store.similarity_search(query, k=k)
    return [doc.page_content for doc in docs]

vector_test = vector_retrieval(
    "Where is Tesla headquartered?",
    k=3
)

for i, item in enumerate(vector_test, 1):
    print(f"\n--- Vector result {i} ---\n{item}")

## 20. Extract graph entities from the query

In [ ]:
query_entity_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Extract important named entities from the user's question.

Return only entities that could reasonably exist as nodes in the knowledge graph.
Do not invent entities.
"""
    ),
    ("human", "{query}")
])

query_entity_llm = llm.with_structured_output(QueryEntities)

def extract_query_entities(query: str) -> list[str]:
    chain = query_entity_prompt | query_entity_llm
    result = chain.invoke({"query": query})
    return [x.strip() for x in result.entities if x.strip()]

print(extract_query_entities(
    "Which company is headquartered in the same city as Tesla?"
))

## 21. Graph retrieval – 1 to 2 hops

In [ ]:
def graph_retrieval(query: str, max_hops: int = 2, limit: int = 40) -> list[dict]:
    entities = extract_query_entities(query)
    results = []

    for entity in entities:
        records = driver.execute_query(
            f"""
            MATCH (start:Entity)
            WHERE toLower(start.name) = toLower($entity)

            MATCH path = (start)-[*1..{max_hops}]-(connected:Entity)

            RETURN
                [node IN nodes(path) | node.name] AS nodes,
                [rel IN relationships(path) | type(rel)] AS relationships
            LIMIT $limit
            """,
            entity=entity,
            limit=limit
        ).records

        for record in records:
            results.append({
                "nodes": record["nodes"],
                "relationships": record["relationships"]
            })

    return results

def graph_context_to_text(graph_results: list[dict]) -> list[str]:
    facts = []

    for item in graph_results:
        nodes = item["nodes"]
        relationships = item["relationships"]

        if not nodes or not relationships:
            continue

        parts = [nodes[0]]
        for rel, node in zip(relationships, nodes[1:]):
            parts.extend([f"--[{rel}]-->", node])

        facts.append(" ".join(parts))

    return list(dict.fromkeys(facts))

graph_results = graph_retrieval(
    "Where is Tesla headquartered?",
    max_hops=2
)

graph_facts = graph_context_to_text(graph_results)

for fact in graph_facts[:20]:
    print(fact)

## 22. Reciprocal Rank Fusion

RRF combines ranked results from vector retrieval and graph retrieval:

`RRF(d) = sum(1 / (k + rank_i(d)))`

In [ ]:
def reciprocal_rank_fusion(
    vector_results: list[str],
    graph_results: list[str],
    k: int = 60
) -> list[tuple[str, float]]:
    scores = {}
    original = {}

    for rank, item in enumerate(vector_results, start=1):
        key = item.strip()
        if not key:
            continue
        scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
        original[key] = item

    for rank, item in enumerate(graph_results, start=1):
        key = item.strip()
        if not key:
            continue
        scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
        original[key] = item

    ranked = sorted(scores.items(), key=lambda pair: pair[1], reverse=True)
    return [(original[key], score) for key, score in ranked]

print(reciprocal_rank_fusion(
    ["A", "B", "C"],
    ["B", "C", "D"]
))

## 23. Hybrid GraphRAG retrieval

In [ ]:
def hybrid_retrieval(
    query: str,
    vector_k: int = 5,
    graph_limit: int = 40,
    final_k: int = 10
):
    vector_results = vector_retrieval(query, k=vector_k)

    graph_results_raw = graph_retrieval(
        query,
        max_hops=2,
        limit=graph_limit
    )
    graph_results = graph_context_to_text(graph_results_raw)

    fused = reciprocal_rank_fusion(
        vector_results,
        graph_results
    )

    return {
        "vector_results": vector_results,
        "graph_results": graph_results,
        "fused_results": fused[:final_k]
    }

hybrid_test = hybrid_retrieval(
    "What company is headquartered in the same city as Tesla?"
)

print("VECTOR RESULTS")
for x in hybrid_test["vector_results"][:3]:
    print("-", x[:250].replace("\n", " "))

print("\nGRAPH RESULTS")
for x in hybrid_test["graph_results"][:10]:
    print("-", x)

print("\nFUSED RESULTS")
for x, score in hybrid_test["fused_results"][:10]:
    print(round(score, 5), "-", x[:250].replace("\n", " "))

## 24. Final answer-generation prompt

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a GraphRAG question-answering assistant.

Answer the user's question using only the supplied context.

The context may contain:
- vector-retrieved document passages
- knowledge-graph facts
- multi-hop paths

Reason across connected facts when necessary.

Requirements:
1. Do not invent information.
2. Give a direct answer.
3. Briefly explain the reasoning chain for multi-hop questions.
4. If the evidence is insufficient, say so.
"""
    ),
    (
        "human",
        """
Question:
{question}

Hybrid context:
{context}

Answer:
"""
    )
])

def answer_from_context(question: str, context_items: list[str]) -> str:
    context = "\n".join(
        f"[Context {i}] {item}"
        for i, item in enumerate(context_items, 1)
    )

    chain = answer_prompt | llm
    response = chain.invoke({
        "question": question,
        "context": context
    })
    return response.content.strip()

## 25. Vector RAG baseline

In [ ]:
def vector_rag_answer(question: str, k: int = 5) -> str:
    contexts = vector_retrieval(question, k=k)
    return answer_from_context(question, contexts)

## 26. GraphRAG answer function

In [ ]:
def graphrag_answer(
    question: str,
    vector_k: int = 5,
    final_k: int = 10
) -> str:
    result = hybrid_retrieval(
        question,
        vector_k=vector_k,
        final_k=final_k
    )

    contexts = [item for item, score in result["fused_results"]]
    return answer_from_context(question, contexts)

## 27. Test the complete pipeline

In [ ]:
simple_question = "Where is Tesla headquartered?"

print("QUESTION:", simple_question)

print("\nVECTOR RAG:")
print(vector_rag_answer(simple_question))

print("\nGRAPHRAG:")
print(graphrag_answer(simple_question))

## 28. Five multi-hop evaluation questions

These questions are designed for the supplied sample document. If you replace the document, update them to match your own knowledge graph.

In [ ]:
evaluation_questions = [
    "Which company is headquartered in the same city as Tesla, and who is a co-founder of that company?",
    "Which organization was founded by the person who is the CEO of Tesla, and where is that organization headquartered?",
    "Which company did Tesla acquire, where was that company headquartered, and who founded it?",
    "Which company is connected to Elon Musk through an acquisition and was later rebranded, and what was its original name?",
    "Which organization is headquartered in Hawthorne, who founded it, and what concept is associated with one of its major systems?"
]

for i, q in enumerate(evaluation_questions, 1):
    print(f"{i}. {q}")

## 29. Run Vector RAG vs GraphRAG comparison

This can take several minutes with a local model because multiple LLM calls are made.

In [ ]:
comparison_rows = []

for i, question in enumerate(evaluation_questions, 1):
    print(f"Running question {i}/{len(evaluation_questions)}...")

    try:
        vector_answer = vector_rag_answer(question)
    except Exception as exc:
        vector_answer = f"ERROR: {exc}"

    try:
        graph_answer = graphrag_answer(question)
    except Exception as exc:
        graph_answer = f"ERROR: {exc}"

    comparison_rows.append({
        "Question": question,
        "Vector RAG": vector_answer,
        "GraphRAG": graph_answer
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## 30. Inspect graph retrieval for every evaluation question

This demonstrates that GraphRAG is actually retrieving graph paths, not just changing the prompt.

In [ ]:
for i, question in enumerate(evaluation_questions, 1):
    print("\n" + "=" * 100)
    print(f"QUESTION {i}: {question}")

    result = hybrid_retrieval(question, vector_k=5, final_k=8)

    print("\nGRAPH FACTS:")
    for fact in result["graph_results"][:10]:
        print("  ", fact)

    print("\nTOP FUSED CONTEXT:")
    for fact, score in result["fused_results"][:8]:
        print(f"  [{score:.5f}] {fact[:300]}")

## 31. Manual evaluation table

Review the five answers and enter 0 or 1:
- Correctness: final answer matches the document.
- Multi-hop: answer follows two or more connected facts.
- Notes: explain why one system performed better or worse.

In [ ]:
evaluation_template = pd.DataFrame({
    "Question": evaluation_questions,
    "Vector RAG Correct (0/1)": [None] * len(evaluation_questions),
    "GraphRAG Correct (0/1)": [None] * len(evaluation_questions),
    "Vector RAG Multi-hop (0/1)": [None] * len(evaluation_questions),
    "GraphRAG Multi-hop (0/1)": [None] * len(evaluation_questions),
    "Notes": [""] * len(evaluation_questions)
})

evaluation_template

## 32. Calculate evaluation scores

Enter 0/1 values in the previous table and rerun this cell.

In [ ]:
score_columns = [
    "Vector RAG Correct (0/1)",
    "GraphRAG Correct (0/1)",
    "Vector RAG Multi-hop (0/1)",
    "GraphRAG Multi-hop (0/1)"
]

scores = evaluation_template[score_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

print("Evaluation summary:")
print(scores.mean().round(3))

if scores.notna().any().any():
    print("\nCorrectness:")
    print("Vector RAG:", scores["Vector RAG Correct (0/1)"].mean())
    print("GraphRAG:", scores["GraphRAG Correct (0/1)"].mean())
else:
    print("Enter 0/1 scores to calculate results.")

## 33. Useful Neo4j Cypher queries

### Count entities
```cypher
MATCH (n:Entity)
RETURN count(n) AS entities
```

### Count relationships
```cypher
MATCH ()-[r]->()
RETURN count(r) AS relationships
```

### Show the complete graph
```cypher
MATCH (n)-[r]->(m)
RETURN n, r, m
```

### Show Elon Musk's neighborhood
```cypher
MATCH p=(n:Entity {name: 'Elon Musk'})-[*1..2]-(m)
RETURN p
```

### Show all organizations
```cypher
MATCH (n:Entity)
WHERE n.type = 'ORGANIZATION'
RETURN n.name
ORDER BY n.name
```

## 34. Export comparison results

In [ ]:
comparison_csv = Path("graphrag_comparison.csv")
comparison_df.to_csv(comparison_csv, index=False, encoding="utf-8")
print("Saved:", comparison_csv.resolve())

## 35. Assignment conclusion

### What was built

**Document ingestion → Chunking → Ollama/Qwen extraction → Neo4j knowledge graph → Vector search + Graph traversal → RRF → Hybrid context → LLM answer**

### Why GraphRAG is different from basic RAG

Traditional vector RAG primarily retrieves semantically similar text. GraphRAG additionally represents entities and relationships explicitly, allowing the system to traverse connected facts. Hybrid retrieval combines both approaches.

### Example reasoning chain

`Elon Musk → founded → SpaceX → headquartered_in → Hawthorne`

A multi-hop question can therefore be answered by following connected facts instead of relying on one semantically similar paragraph.

### Final checklist

- [x] Neo4j connection
- [x] Knowledge-rich document
- [x] Chunking
- [x] Ollama structured extraction
- [x] Entity extraction
- [x] Relationship extraction
- [x] Triple representation
- [x] Neo4j nodes
- [x] Neo4j relationships
- [x] Graph verification
- [x] Vector retrieval
- [x] Query entity extraction
- [x] 1–2 hop graph traversal
- [x] Reciprocal Rank Fusion
- [x] Hybrid GraphRAG
- [x] Vector RAG baseline
- [x] Five multi-hop questions
- [x] Comparison
- [x] Evaluation template
- [x] CSV export

## 36. Close the Neo4j connection

Run this after you finish the notebook.

In [ ]:
driver.close()
print("Neo4j driver closed.")